In [1]:
import pandas as pd
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from sklearn.preprocessing import LabelEncoder
import numpy as np
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout

In [4]:
import tensorflow as tf

gpu_devices = tf.config.list_physical_devices('GPU')
if gpu_devices:
    print(f"GPU Enabled: {gpu_devices}")
    for gpu in gpu_devices:
        tf.config.experimental.set_memory_growth(gpu, True)
else:
    print("GPU could not found, Tasks will run on CPU")

GPU Enabled: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [5]:
df = pd.read_csv("data/tr_email_spam.csv")

In [7]:
# Sütun isimlerini ve içindeki benzersiz değerleri kontrol et
print("Sütunlar:", df.columns)
print("Classification sütunundaki değerler:\n", df['Classification'].value_counts())

Sütunlar: Index(['Text', 'Classification', 'spam'], dtype='str')
Classification sütunundaki değerler:
 Classification
non-spam    497
spam        329
Name: count, dtype: int64


In [8]:
df = df.sample(frac=1, random_state=42).reset_index(drop=True)# Sonra texts, labels, sequences işlemlerini tekrar yap.

In [9]:
# 1. Eşlemeyi veri setindeki gerçek değerlere göre yapalım
# 'non-spam' -> 0, 'spam' -> 1
df['label'] = df['Classification'].str.strip().str.lower().map({
    'non-spam': 0,
    'spam': 1
})

# 2. Kontrol et (Burada 0: 497, 1: 329 görmelisin)
print("Yeni etiket dağılımı:\n", df['label'].value_counts())

# 3. Veriyi tekrar hazırla
df = df.dropna(subset=['label', 'Text'])
texts = df['Text'].values
labels = df['label'].values

# 4. Tokenizer ve Model eğitimine devam et

Yeni etiket dağılımı:
 label
0    497
1    329
Name: count, dtype: int64


In [10]:
import numpy as np
print("Etiket tipi:", labels.dtype)
print("Sınıf dağılımı:", np.unique(labels, return_counts=True))
# Çıktıda (array([0, 1]), array([497, 329])) görmelisin.

Etiket tipi: int64
Sınıf dağılımı: (array([0, 1]), array([497, 329]))


In [9]:
import pandas as pd
print(df.isnull().sum()) # Eğer pandas kullanıyorsan

df = df.dropna(subset=['Text'])
texts = df['Text'].astype(str).values



Text                0
Classification      0
spam              329
label               0
dtype: int64


In [13]:
tokenizer = Tokenizer(lower=True) # Her şeyi küçük harfe çevirir
tokenizer.fit_on_texts(df['Text'])
X = pad_sequences(tokenizer.texts_to_sequences(df['Text']), maxlen=100)
labels = df['label'].values

In [11]:
import numpy as np
print(np.unique(labels, return_counts=True))

(array([0, 1]), array([497, 329]))


In [14]:
from sklearn.model_selection import train_test_split
from sklearn.utils import class_weight

# 1. Veriyi ve Etiketleri Karıştırarak Böl (Stratify ile sınıfları eşit dağıt)
X_train, X_test, y_train, y_test = train_test_split(
    X, labels, test_size=0.2, random_state=42, stratify=labels
)

# 2. Sınıf Ağırlıklarını Hesapla (Modelin Spam'e daha çok değer vermesini sağlar)
weights = class_weight.compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)
class_weights = {i: weights[i] for i in range(len(weights))}

In [15]:
from tensorflow.keras.layers import GlobalAveragePooling1D

model = Sequential([
    Embedding(input_dim=len(tokenizer.word_index) + 1, output_dim=16), # Düşük boyut ezberi önler
    GlobalAveragePooling1D(),
    Dense(16, activation='relu'),
    Dropout(0.2),
    Dense(1, activation='sigmoid')
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.005),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# 4. Eğit (Epoch sayısını artırıp batch_size'ı küçültüyoruz)
print("Eğitim Başlıyor...")
model.fit(
    X_train, y_train,
    epochs=100,
    batch_size=16,
    validation_data=(X_test, y_test),
    class_weight=class_weights, # Spam'leri görmezden gelmesini engeller
    verbose=0
)
print("Eğitim Tamamlandı!")

2026-03-17 11:05:13.817283: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M4
2026-03-17 11:05:13.817315: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 16.00 GB
2026-03-17 11:05:13.817321: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 5.92 GB
2026-03-17 11:05:13.817339: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2026-03-17 11:05:13.817351: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


Eğitim Başlıyor...


2026-03-17 11:05:14.108049: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.


Eğitim Tamamlandı!


In [16]:
import numpy as np

# Random texts for testing mail.
random_mails = [
    "Merhaba, 2026 MWC konferansına davetlisiniz.",
    "BAK buraya seni or*spu çocuğu. 100$ BAhis oyna para kazan"
]

random_mails = list(map(lambda x: x.lower(), random_mails))



# Tokenize mails
new_seq = tokenizer.texts_to_sequences(random_mails)
new_pad = pad_sequences(new_seq, maxlen=100)

# Prediction
preds = model.predict(new_pad)

# Results
for i, mail in enumerate(random_mails):
    prob = preds[i][0]
    state = ("KATIKSIZ SPAM" if prob > 0.75 else "SPAM") if prob > 0.5 else "SECURE"
    print(f"\nMail: {mail}")
    print(f"{state} (Probability: %{prob*100:.2f})")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step

Mail: merhaba, 2026 mwc konferansına davetlisiniz.
SECURE (Probability: %3.54)

Mail: bak buraya seni orospu çocuğu. 100$ bahis oyna para kazan
SECURE (Probability: %9.65)


In [ ]:
# Save model to a file
model.save('spam_model_v3.h5')

In [ ]:
import pickle
with open('tokenizer3.pickle', 'wb') as handle:
    pickle.dump(tokenizer, handle, protocol=pickle.HIGHEST_PROTOCOL)